# RNN 實戰：字符級文本生成

> **項目目標**: 使用 LSTM 生成莎士比亞風格的文本
> 
> **難度**: ⭐⭐⭐ 中級
> 
> **預計時間**: 2-3 小時

## 📋 項目大綱

1. 數據準備
2. 字符級編碼
3. 序列生成模型
4. 訓練策略
5. 文本生成
6. 溫度採樣
7. 多樣性控制
8. AI 輔助改進

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import random

# 設置隨機種子
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用設備: {device}')

## 1. 數據準備

我們將使用莎士比亞的作品作為訓練數據

In [ ]:
# 示例文本（實際使用時應該下載莎士比亞全集）
sample_text = """
To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them. To die—to sleep,
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to: 'tis a consummation
Devoutly to be wish'd. To die, to sleep;
To sleep, perchance to dream—ay, there's the rub:
For in that sleep of death what dreams may come,
When we have shuffled off this mortal coil,
Must give us pause—there's the respect
That makes calamity of so long life.
"""

# 或者生成更長的示例數據
def generate_longer_text(seed_text, length=10000):
    """生成更長的訓練文本"""
    words = seed_text.split()
    generated = []
    for i in range(length):
        generated.append(random.choice(words))
    return ' '.join(generated)

# 使用示例文本或生成更長的文本
text = sample_text * 100  # 重複文本以增加長度

print(f"文本長度: {len(text)} 字符")
print(f"前 200 個字符:\n{text[:200]}...")

## 2. 字符級編碼

In [ ]:
class CharVocab:
    """字符級詞表"""
    def __init__(self, text):
        self.chars = sorted(list(set(text)))
        self.char_to_idx = {ch: i for i, ch in enumerate(self.chars)}
        self.idx_to_char = {i: ch for i, ch in enumerate(self.chars)}
        self.vocab_size = len(self.chars)
    
    def encode(self, text):
        return [self.char_to_idx[ch] for ch in text]
    
    def decode(self, indices):
        return ''.join([self.idx_to_char[i] for i in indices])

# 創建詞表
vocab = CharVocab(text)
print(f"詞表大小: {vocab.vocab_size}")
print(f"字符集: {''.join(vocab.chars)}")

# 編碼文本
encoded_text = vocab.encode(text)
print(f"\n編碼前: {text[:50]}")
print(f"編碼後: {encoded_text[:50]}")

## 3. 數據集準備

In [ ]:
def create_sequences(text, seq_length=100):
    """創建訓練序列"""
    sequences = []
    targets = []
    
    for i in range(len(text) - seq_length):
        seq = text[i:i+seq_length]
        target = text[i+1:i+seq_length+1]
        sequences.append(seq)
        targets.append(target)
    
    return sequences, targets

# 創建序列
seq_length = 100
sequences, targets = create_sequences(encoded_text, seq_length)

print(f"序列數量: {len(sequences)}")
print(f"序列長度: {seq_length}")
print(f"\n第一個序列:")
print(f"輸入: {vocab.decode(sequences[0])}")
print(f"目標: {vocab.decode(targets[0])}")

## 4. 字符級 LSTM 模型

In [ ]:
class CharLSTM(nn.Module):
    """字符級 LSTM 語言模型"""
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.3):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 嵌入層
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        
        # LSTM 層
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        
        # 輸出層
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden=None):
        # x: (batch, seq_len)
        embedded = self.dropout(self.embedding(x))
        
        # LSTM
        if hidden is None:
            lstm_out, hidden = self.lstm(embedded)
        else:
            lstm_out, hidden = self.lstm(embedded, hidden)
        
        # 輸出
        lstm_out = self.dropout(lstm_out)
        output = self.fc(lstm_out)
        
        return output, hidden
    
    def init_hidden(self, batch_size, device):
        """初始化隱藏狀態"""
        h = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        c = torch.zeros(self.num_layers, batch_size, self.hidden_dim).to(device)
        return (h, c)

# 創建模型
model = CharLSTM(
    vocab_size=vocab.vocab_size,
    embed_dim=128,
    hidden_dim=256,
    num_layers=2,
    dropout=0.3
).to(device)

print(model)
print(f"\n參數量: {sum(p.numel() for p in model.parameters()):,}")

## 5. 訓練函數

In [ ]:
def train_epoch(model, sequences, targets, batch_size, criterion, optimizer, device):
    """訓練一個 epoch"""
    model.train()
    total_loss = 0
    num_batches = len(sequences) // batch_size
    
    # 打亂數據
    indices = np.random.permutation(len(sequences))
    
    pbar = tqdm(range(num_batches), desc="Training")
    for i in pbar:
        # 獲取批次
        batch_indices = indices[i*batch_size:(i+1)*batch_size]
        batch_seqs = torch.tensor([sequences[j] for j in batch_indices]).to(device)
        batch_targets = torch.tensor([targets[j] for j in batch_indices]).to(device)
        
        # 前向傳播
        hidden = model.init_hidden(batch_size, device)
        outputs, hidden = model(batch_seqs, hidden)
        
        # 計算損失
        loss = criterion(outputs.view(-1, vocab.vocab_size), batch_targets.view(-1))
        
        # 反向傳播
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})
    
    return total_loss / num_batches

# 訓練設置
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

batch_size = 128
num_epochs = 20

## 6. 文本生成函數

In [ ]:
def generate_text(model, vocab, seed_text, length=200, temperature=1.0, device='cpu'):
    """生成文本
    
    Args:
        model: 訓練好的模型
        vocab: 詞表
        seed_text: 種子文本
        length: 生成長度
        temperature: 溫度參數（控制隨機性）
            - temperature < 1: 更確定性（保守）
            - temperature = 1: 標準採樣
            - temperature > 1: 更隨機（創造性）
    """
    model.eval()
    
    # 編碼種子文本
    current_seq = vocab.encode(seed_text)
    generated = seed_text
    
    # 初始化隱藏狀態
    hidden = model.init_hidden(1, device)
    
    with torch.no_grad():
        for _ in range(length):
            # 準備輸入
            x = torch.tensor([current_seq]).to(device)
            
            # 預測
            output, hidden = model(x, hidden)
            
            # 獲取最後一個時間步的輸出
            logits = output[0, -1, :] / temperature
            probs = F.softmax(logits, dim=0)
            
            # 採樣
            next_char_idx = torch.multinomial(probs, 1).item()
            next_char = vocab.idx_to_char[next_char_idx]
            
            # 更新序列
            generated += next_char
            current_seq = current_seq[1:] + [next_char_idx]
    
    return generated


def generate_with_topk(model, vocab, seed_text, length=200, k=5, device='cpu'):
    """使用 Top-K 採樣生成文本"""
    model.eval()
    
    current_seq = vocab.encode(seed_text)
    generated = seed_text
    hidden = model.init_hidden(1, device)
    
    with torch.no_grad():
        for _ in range(length):
            x = torch.tensor([current_seq]).to(device)
            output, hidden = model(x, hidden)
            
            logits = output[0, -1, :]
            
            # Top-K 採樣
            top_k_logits, top_k_indices = torch.topk(logits, k)
            probs = F.softmax(top_k_logits, dim=0)
            
            # 從 top-k 中採樣
            sampled_idx = torch.multinomial(probs, 1).item()
            next_char_idx = top_k_indices[sampled_idx].item()
            next_char = vocab.idx_to_char[next_char_idx]
            
            generated += next_char
            current_seq = current_seq[1:] + [next_char_idx]
    
    return generated

## 7. 開始訓練

In [ ]:
losses = []
generated_samples = []

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-" * 60)
    
    # 訓練
    loss = train_epoch(model, sequences, targets, batch_size, criterion, optimizer, device)
    losses.append(loss)
    
    # 學習率調整
    scheduler.step()
    
    print(f"Loss: {loss:.4f}")
    
    # 每 5 個 epoch 生成樣本
    if (epoch + 1) % 5 == 0:
        print("\n生成樣本:")
        seed = "To be"
        sample = generate_text(model, vocab, seed, length=200, temperature=0.8, device=device)
        print(sample)
        generated_samples.append((epoch+1, sample))

# 保存模型
torch.save(model.state_dict(), 'char_lstm_model.pth')
print("\n✓ 模型已保存！")

## 8. 訓練曲線可視化

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(losses, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.grid(True)
plt.savefig('training_loss.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"最終損失: {losses[-1]:.4f}")
print(f"初始損失: {losses[0]:.4f}")
print(f"降低: {(1 - losses[-1]/losses[0])*100:.2f}%")

## 9. 溫度參數實驗

In [ ]:
seed_text = "To be, or not to be"
temperatures = [0.5, 0.8, 1.0, 1.2, 1.5]

print("\n" + "=" * 80)
print("溫度參數對比實驗")
print("=" * 80)

for temp in temperatures:
    print(f"\n溫度 = {temp}:")
    print("-" * 80)
    generated = generate_text(model, vocab, seed_text, length=200, temperature=temp, device=device)
    print(generated)
    print("-" * 80)

## 10. Top-K 採樣對比

In [ ]:
print("\n" + "=" * 80)
print("Top-K 採樣對比")
print("=" * 80)

k_values = [3, 5, 10, 20]

for k in k_values:
    print(f"\nTop-K = {k}:")
    print("-" * 80)
    generated = generate_with_topk(model, vocab, seed_text, length=200, k=k, device=device)
    print(generated)
    print("-" * 80)

## 11. 互動式文本生成

In [ ]:
def interactive_generation():
    """互動式文本生成"""
    print("\n" + "=" * 80)
    print("互動式文本生成器")
    print("=" * 80)
    print("輸入 'quit' 退出\n")
    
    while True:
        seed = input("\n請輸入種子文本: ").strip()
        
        if seed.lower() == 'quit':
            print("再見！👋")
            break
        
        if not seed:
            print("請輸入有效的文本")
            continue
        
        try:
            length = int(input("生成長度 (預設 200): ") or "200")
            temp = float(input("溫度 (0.5-1.5, 預設 1.0): ") or "1.0")
            
            print("\n生成中...\n")
            generated = generate_text(model, vocab, seed, length=length, temperature=temp, device=device)
            
            print("-" * 80)
            print(generated)
            print("-" * 80)
            
        except ValueError:
            print("無效的輸入，請重試")
        except Exception as e:
            print(f"錯誤: {e}")

# 運行互動式生成
# interactive_generation()

## 12. 生成質量評估

In [ ]:
def evaluate_diversity(texts):
    """評估生成文本的多樣性"""
    unique_chars = set(''.join(texts))
    unique_words = set(' '.join(texts).split())
    
    return {
        'unique_chars': len(unique_chars),
        'unique_words': len(unique_words),
        'avg_length': np.mean([len(t) for t in texts])
    }

# 生成多個樣本進行評估
samples = []
seeds = ["To be", "The time", "Love is", "When shall", "All the"]

for seed in seeds:
    sample = generate_text(model, vocab, seed, length=200, temperature=1.0, device=device)
    samples.append(sample)

metrics = evaluate_diversity(samples)
print("\n生成文本多樣性評估:")
print(f"  唯一字符數: {metrics['unique_chars']}")
print(f"  唯一詞語數: {metrics['unique_words']}")
print(f"  平均長度: {metrics['avg_length']:.2f}")

print("\n生成樣本:")
for i, sample in enumerate(samples[:3], 1):
    print(f"\n樣本 {i}:")
    print("-" * 80)
    print(sample)
    print("-" * 80)

## 13. AI 輔助改進建議

### 使用 ChatGPT/Claude 改進生成質量

#### 提示詞模板 1: 分析生成質量
```
我訓練了一個字符級 LSTM 模型來生成莎士比亞風格的文本。
以下是模型生成的樣本：

[粘貼生成的文本]

請分析：
1. 語法和語義的連貫性
2. 風格是否接近莎士比亞
3. 存在的主要問題
4. 如何改進模型
```

#### 提示詞模板 2: 超參數建議
```
我的字符級 LSTM 模型配置如下：
- 嵌入維度: 128
- 隱藏層維度: 256
- 層數: 2
- Dropout: 0.3
- 序列長度: 100

訓練數據大小: [數據大小]
當前困惑度: [困惑度值]

請建議：
1. 如何調整超參數提高性能
2. 是否需要更複雜的架構（如 Attention）
3. 數據增強策略
```


## 14. 進階技術

### 1. 詞級生成（更連貫）

In [ ]:
# 詞級模型架構（示例）
class WordLSTM(nn.Module):
    """詞級 LSTM 語言模型"""
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=512, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        if hidden is None:
            lstm_out, hidden = self.lstm(embedded)
        else:
            lstm_out, hidden = self.lstm(embedded, hidden)
        output = self.fc(lstm_out)
        return output, hidden

print("詞級模型適合生成更連貫的文本")
print("優點: 語法更好、訓練更快")
print("缺點: 需要更大的詞表、無法生成新詞")

### 2. 添加注意力機制

In [ ]:
class AttentionLSTM(nn.Module):
    """帶自注意力的 LSTM"""
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        
        # 自注意力
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads=4, batch_first=True)
        
        self.fc = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, x, hidden=None):
        embedded = self.embedding(x)
        lstm_out, hidden = self.lstm(embedded, hidden)
        
        # 應用自注意力
        attn_out, _ = self.attention(lstm_out, lstm_out, lstm_out)
        
        output = self.fc(attn_out)
        return output, hidden

print("注意力機制可以幫助模型關注重要的上下文信息")

## 15. 總結和改進方向

### 本項目學到的內容

1. ✅ 字符級語言建模
2. ✅ LSTM 序列生成
3. ✅ 溫度採樣和 Top-K 採樣
4. ✅ 文本生成技巧

### 改進建議

1. **數據增強**
   - 使用更大的語料庫
   - 清洗和預處理數據

2. **模型改進**
   - 嘗試 GRU（更快）
   - 添加注意力機制
   - 使用 Transformer

3. **訓練技巧**
   - 學習率調度
   - Early Stopping
   - 模型集成

4. **評估指標**
   - 困惑度（Perplexity）
   - BLEU 分數
   - 人工評估

### 延伸項目

- 代碼生成（Python/JavaScript）
- 歌詞生成
- 對話生成
- 詩歌創作

---

**恭喜完成文本生成項目！🎉**
